# Get to know DynaCell v1

Can a model look at a living cell under plain light and recover the organelles it cannot see? [DynaCell v1](https://registry.opendata.aws/dynacell) is the benchmark built to answer that. It pairs label-free 3D microscopy (quantitative phase and brightfield) with fluorescence images of tagged organelles, the target a *virtual staining* model learns to reproduce.

The release spans two cell types (A549 lung epithelial cells and hiPSCs), four organelles (plasma membrane, chromatin, ER, mitochondria), and three infection states in A549 (mock, Dengue, Zika), about 677 GB. Rather than download any of it, this notebook **streams** slices straight from S3, fetching only the bytes it shows. It has three parts:

1. **What's in the bucket** explores the data layout and metadata.
2. **Data diversity** shows the data across channels, conditions, and cell types.
3. **Open challenge** frames virtual staining that generalizes across cell types and perturbations.

Find more tutorials and documentation at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

## 1. What's in the bucket

The public `dynacell` S3 bucket keeps everything under the `v1/` prefix:

| Prefix | Contents |
|---|---|
| `v1/data/` | OME-Zarr imaging archives, the dataset itself |
| `v1/models/` | Pre-trained virtual-staining checkpoints |
| `v1/demo/` | A <1 GB reviewer bundle for a quick taste |
| `v1/metadata/` · `v1/supplementary/` · `v1/README.md` | Croissant metadata, extras, and the top-level README |

Imaging archives split by source. **`biohub-a549/`** (387 GB, CC-BY-4.0) holds 24 files: four organelles (`CAAX` membrane, `H2B` chromatin, `SEC61B` ER, `TOMM20` mitochondria) × three conditions (`mock`, `DENV`, `ZIKV`) × two splits. **`aics-hipsc/`** (290 GB, Allen Institute terms) adds a second cell type with the same four organelles, but no infection conditions. Each file is one self-contained OME-Zarr plate. We list them with `boto3`, which needs no download.

In [ ]:
# Requires: fsspec[http]>=2024, zarr>=3, boto3>=1.38, numpy>=2, matplotlib>=3.9
# The .ozx is iohub's RFC-9 zipped OME-Zarr; we stream it with fsspec + zarr, iohub's storage engine.
# Install with pip, conda, or uv as suits your environment.

import json

import boto3
import fsspec
import matplotlib.pyplot as plt
import numpy as np
import zarr
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "dynacell"
PREFIX = "v1/"
HTTPS_BASE = "https://dynacell.s3.us-west-2.amazonaws.com/v1/"

# Anonymous client for listing the public bucket (no download, no credentials).
s3 = boto3.client("s3", region_name="us-west-2", config=Config(signature_version=UNSIGNED))

# Each .ozx is a single ZIP behind one file handle; serialize zarr's async reads to share it safely.
zarr.config.set({"async.concurrency": 1})


def open_ozx(rel_path):
    """Stream a remote ``.ozx`` as a Zarr v3 group over HTTPS, fetching only the bytes touched.

    Returns ``(mapper, group)``: ``mapper`` reads JSON metadata keys (e.g. ``mapper["zarr.json"]``);
    ``group`` gives array access where each slice pulls just its shard via HTTP range requests.
    """
    mapper = fsspec.get_mapper(f"zip://::{HTTPS_BASE}{rel_path}")
    group = zarr.open_group(mapper, mode="r", zarr_format=3)
    return mapper, group


def fov_paths(mapper):
    """List the FOV array paths (``<row>/<col>/<fov>``) inside an opened ``.ozx``."""
    suffix = "/0/zarr.json"
    return sorted({k[: -len(suffix)] for k in mapper if k.endswith(suffix) and k.count("/") == 4})

In [ ]:
# List top-level prefixes under v1/
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")

print("Directories:")
for p in resp.get("CommonPrefixes", []):
    print(" ", p["Prefix"])

print("\nFiles:")
for o in resp.get("Contents", []):
    print(f"  {o['Key']}  ({o['Size']:,} bytes)")

In [ ]:
# Test-split archives for both cell types
for prefix in ("v1/data/biohub-a549/test/", "v1/data/aics-hipsc/test/"):
    print(prefix)
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    for o in resp.get("Contents", []):
        name = o["Key"].split("/")[-1]
        if name:  # skip the directory entry itself
            print(f"    {name:<20} {o['Size'] / 1e9:>6.1f} GB")

### The `.ozx` format, and reading metadata without downloading

Each archive is a single `.ozx` file: a ZIP wrapping a [Zarr v3](https://zarr.readthedocs.io/) store in the [OME-NGFF](https://ngff.openmicroscopy.org/latest/) layout (iohub's RFC-9 *zipped OME-Zarr*). Because S3 serves HTTP range requests, we mount the remote ZIP with `fsspec` and read it with `zarr`, pulling only the shards we touch. A full archive is 8 to 13 GB; the cells below read its metadata, and later a few slices, fetching mere megabytes. (To work with a downloaded archive locally instead, `iohub.open_ome_zarr` reads `.ozx` directly.)

Inside is an HCS plate: a single well of fields of view (FOVs), each a 5D `(T, C, Z, Y, X)` array of timepoints, channels, and a 3D volume. Every plate also carries provenance (`assembly_*` keys), channel names, voxel spacing, and per-channel normalization statistics that virtual-staining models train against. Let's open the A549 ER (SEC61B) mock archive and read all of it, streamed.

In [ ]:
# Open one archive's metadata by streaming, with no full download.
REL = "data/biohub-a549/test/SEC61B_mock.ozx"
mapper, group = open_ozx(REL)

fovs = fov_paths(mapper)
arr = group[f"{fovs[0]}/0"]
print(f"{len(fovs)} FOVs, each {arr.shape} (T, C, Z, Y, X), dtype {arr.dtype}")

In [ ]:
# Plate-level metadata lives in the root zarr.json, read straight from the archive.
root_attrs = json.loads(mapper["zarr.json"])["attributes"]

print("=== Provenance ===")
for key in (
    "assembly_target",
    "assembly_condition",
    "assembly_split",
    "assembly_center_crop_yx",
    "assembly_t_cap",
    "assembly_target_yx_pixel_size_um",
):
    print(f"  {key}: {root_attrs[key]}")

print("\n=== Per-channel normalization (dataset percentiles) ===")
for ch, stats in root_attrs["normalization"].items():
    ds = stats["dataset_statistics"]
    print(f"  {ch:<12} p1={ds['p1']:.3f}  p99={ds['p99']:.3f}  mean={ds['mean']:.3f}")

In [ ]:
# Channel names and voxel spacing live in each FOV's zarr.json.
fov_attrs = json.loads(mapper[f"{fovs[0]}/zarr.json"])["attributes"]["ome"]
channels = [c["label"] for c in fov_attrs["omero"]["channels"]]
print("Channels:", channels)

ms = fov_attrs["multiscales"][0]
scale = ms["datasets"][0]["coordinateTransformations"][0]["scale"]
print("\nAxis : voxel size")
for ax, s in zip(ms["axes"], scale):
    print(f"  {ax['name']}: {s} {ax.get('unit', '-')}")

## 2. Data diversity: channels, conditions, and cell types

DynaCell's value is its breadth: the same virtual-staining task posed across **channels** (the label-free brightfield and phase inputs, and the fluorescence target), across **infection conditions** in A549 (mock, Dengue, Zika), and across **cell types** (hiPSC and A549). We show one grid per organelle target. Each grid is three rows (Brightfield, Phase3D, fluorescence) by four columns: the left two are unperturbed cells (hiPSC, then A549 mock), the right two are A549 perturbed by Dengue and Zika.

Each panel is the central Z-slice of a random field of view. The fluorescence panels lift their shadows with a display gamma, so dim structure and the organelle-free voids (such as nuclei in the ER and mitochondria channels) stay visible. These are flat slices of 3D volumes over time; explore the full volumes, timepoints, and model predictions interactively in the [DynaCell HuggingFace demo](https://huggingface.co/spaces/biohub/dynacell). All four organelles (membrane, nuclei, ER, mitochondria) are present for both cell types; A549 adds the three infection conditions. We stream only the slices each grid needs.

In [ ]:
# One grid per organelle target. Across cell types the fluorescence channel keeps a single label
# ("Membrane", "Nuclei", or "Structure" for the endogenously tagged organelle); only the archive changes.
# Each tuple: (organelle display name, fluorescence channel, display gamma, [(column label, archive)]).
GRIDS = [
    (
        "Membrane",
        "Membrane",
        0.7,
        [
            ("hiPSC", "data/aics-hipsc/test/cell.ozx"),
            ("A549 mock", "data/biohub-a549/test/CAAX_mock.ozx"),
            ("A549 DENV", "data/biohub-a549/test/CAAX_DENV.ozx"),
            ("A549 ZIKV", "data/biohub-a549/test/CAAX_ZIKV.ozx"),
        ],
    ),
    (
        "Nuclei",
        "Nuclei",
        0.85,
        [
            ("hiPSC", "data/aics-hipsc/test/cell.ozx"),
            ("A549 mock", "data/biohub-a549/test/H2B_mock.ozx"),
            ("A549 DENV", "data/biohub-a549/test/H2B_DENV.ozx"),
            ("A549 ZIKV", "data/biohub-a549/test/H2B_ZIKV.ozx"),
        ],
    ),
    (
        "ER",
        "Structure",
        0.6,
        [
            ("hiPSC", "data/aics-hipsc/test/SEC61B.ozx"),
            ("A549 mock", "data/biohub-a549/test/SEC61B_mock.ozx"),
            ("A549 DENV", "data/biohub-a549/test/SEC61B_DENV.ozx"),
            ("A549 ZIKV", "data/biohub-a549/test/SEC61B_ZIKV.ozx"),
        ],
    ),
    (
        "Mitochondria",
        "Structure",
        0.6,
        [
            ("hiPSC", "data/aics-hipsc/test/TOMM20.ozx"),
            ("A549 mock", "data/biohub-a549/test/TOMM20_mock.ozx"),
            ("A549 DENV", "data/biohub-a549/test/TOMM20_DENV.ozx"),
            ("A549 ZIKV", "data/biohub-a549/test/TOMM20_ZIKV.ozx"),
        ],
    ),
]

rng = np.random.default_rng(7)  # reproducible "random" FOV per panel


def normalize(plane, p1, p99, gamma=1.0):
    """Clip to the archive's stored percentile bounds, scale to [0, 1], then apply display gamma."""
    scaled = np.clip((plane.astype(float) - p1) / (p99 - p1 + 1e-9), 0.0, 1.0)
    return scaled**gamma if gamma != 1.0 else scaled


def column_panels(rel_path, organelle, channel, gamma):
    """Stream the central Z-slice of a random FOV: brightfield, phase, and the organelle fluorescence."""
    mapper, group = open_ozx(rel_path)
    norm = json.loads(mapper["zarr.json"])["attributes"]["normalization"]
    fovs = fov_paths(mapper)
    fov = fovs[int(rng.integers(len(fovs)))]
    channels = [c["label"] for c in json.loads(mapper[f"{fov}/zarr.json"])["attributes"]["ome"]["omero"]["channels"]]
    arr = group[f"{fov}/0"]
    z = arr.shape[2] // 2
    rows = {"Brightfield": ("Brightfield", 1.0), "Phase3D": ("Phase3D", 1.0), organelle: (channel, gamma)}
    panels = {}
    for row, (ch, gm) in rows.items():
        ds = norm[ch]["dataset_statistics"]
        panels[row] = normalize(np.asarray(arr[0, channels.index(ch), z]), ds["p1"], ds["p99"], gm)
    return panels, fov.split("/")[-1]


def diversity_grid(organelle, channel, gamma, columns):
    """Stream and plot the 3x4 (input rows x cell-type/condition columns) grid for one organelle."""
    panels = {label: column_panels(rel, organelle, channel, gamma) for label, rel in columns}
    rows = ["Brightfield", "Phase3D", organelle]
    cmaps = {"Brightfield": "gray", "Phase3D": "gray", organelle: "inferno"}
    fig, axes = plt.subplots(3, 4, figsize=(16, 13), facecolor="black")
    for j, (label, _) in enumerate(columns):
        col_panels, fov = panels[label]
        for i, row in enumerate(rows):
            ax = axes[i, j]
            ax.imshow(col_panels[row], cmap=cmaps[row], vmin=0, vmax=1)
            ax.axis("off")
            if i == 0:
                ax.set_title(f"{label}\n({fov})", color="white", fontsize=12, pad=6)
            if j == 0:
                ax.text(
                    -0.06,
                    0.5,
                    row,
                    transform=ax.transAxes,
                    color="white",
                    fontsize=13,
                    rotation=90,
                    va="center",
                    ha="center",
                )
    # Reserve the top band for the figure title so it clears the column headers.
    fig.subplots_adjust(left=0.05, right=0.99, top=0.88, bottom=0.01, wspace=0.04, hspace=0.06)
    fig.suptitle(
        f"DynaCell {organelle}: brightfield and phase inputs with fluorescence target, across cell type and infection",
        color="white",
        fontsize=14,
        y=0.955,
    )
    plt.show()

In [ ]:
# One streamed 3x4 grid per organelle target.
for organelle, channel, gamma, columns in GRIDS:
    diversity_grid(organelle, channel, gamma, columns)

## 3. Open challenge: virtual staining that generalizes across cell types and perturbations

The diversity above is also the hard part. *Can one virtual-staining protocol predict organelle morphology in a cell type it was not trained on, and across infections that reshape the very organelles it must recover?* A model fit to unperturbed A549 cells may falter on hiPSCs, or on Dengue- and Zika-infected cells where membranes, ER, and mitochondria reorganize. Closing that gap is the benchmark's purpose.

Ways in:

- **Run the benchmark.** The dataset ships with eval configs and a pipeline (`dynacell evaluate`, `dynacell evaluate-grouped`) that scores predictions on pixel similarity (Spectral PCC, SSIM) and cell-level features (DINOv3, DynaCLR) for each (model, organelle, condition).
- **Train and test across the axes.** Hold out a cell type or a condition; the drop in score is the generalization gap. Joint training across A549 and hiPSC, and across organelles, is one lever; domain adaptation from unpaired brightfield is another.
- **Start from a baseline.** Pre-trained VSCyto3D, FNet3D, UNeXt2, and CELL-Diff weights live under `v1/models/`.

Code and configs: https://github.com/mehta-lab/VisCy/tree/dynacell-models/